# 04 · Content fetch

## Why

Everything modelled so far used **titles**. A title is a nine-word label, not the content. An R²
of 0.04 on titles says titles do not predict performance; it says nothing about whether content
does. Treating those as the same claim was an error in an earlier draft.

Content sits **upstream** of distribution — what gets made drives click-through, watch time and
retention, and those drive how far a video is pushed. So the content question cannot be dismissed
by showing that a weak proxy for it has no explanatory power.

This notebook collects two richer representations and reports, honestly, how much of each is
obtainable.

**Run it top to bottom.** Everything is cached to disk, so re-running costs nothing.

---
`pip install yt-dlp curl-cffi google-api-python-client python-dotenv pandas`

In [ ]:
import os, json, time, random, pathlib, re, subprocess, shutil, tempfile
from collections import Counter
import pandas as pd, numpy as np
from dotenv import load_dotenv

CWD = pathlib.Path.cwd()
NB = next((p for p in [CWD, CWD/"notebook", *CWD.parents] if (p/"data").is_dir()), CWD)
DATA = NB/"data"; RAW = DATA/"content"; RAW.mkdir(exist_ok=True)
DESC_CACHE, TR_CACHE = RAW/"descriptions.jsonl", RAW/"transcripts.jsonl"

load_dotenv(NB/".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
assert API_KEY, "YOUTUBE_API_KEY missing from notebook/.env"

vids = pd.read_csv(DATA/"videos_clean.csv", parse_dates=["date"])
print(f"{len(vids):,} videos on file · {vids.date.min().date()} → {vids.date.max().date()}")

## 1. Target window

The question is about the period after reach fell, so the target is that period plus a comparable
one before it. The podcast is excluded — `02` established it is a different product.

In [ ]:
WINDOWS = {"pre_decline":  ("2021-01-01", "2023-06-30"),
           "post_decline": ("2023-07-01", "2026-06-30")}
CHANNELS = ["LinusTechTips", "Techquickie", "ShortCircuit", "TechLinked"]

sel = vids[vids.channel.isin(CHANNELS) & (vids.format != "wan_show")].copy()
sel["window"] = np.select(
    [sel.date.between(*WINDOWS["pre_decline"]), sel.date.between(*WINDOWS["post_decline"])],
    ["pre_decline", "post_decline"], default=None)
target = sel[sel.window.notna()].reset_index(drop=True)

print(target.pivot_table(index="channel", columns="window", values="id",
                         aggfunc="count", fill_value=0).to_string())
print(f"\n{len(target):,} videos in scope")

## 2. Descriptions — official API

`videos.list(part=snippet)`, the same endpoint used in `01`. One quota unit per fifty videos.
This part is reliable and completes for the whole target set.

In [ ]:
def cached_ids(path, statuses=None):
    if not path.exists():
        return set()
    out = set()
    for l in path.open():
        if not l.strip():
            continue
        r = json.loads(l)
        if statuses is None or r.get("status") in statuses:
            out.add(r["id"])
    return out

todo = [i for i in target.id if i not in cached_ids(DESC_CACHE)]
print(f"{len(target)-len(todo):,} cached · {len(todo):,} to fetch")

if todo and not globals().get("OFFLINE_DESC", False):
    from googleapiclient.discovery import build
    api = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
    with DESC_CACHE.open("a") as f:
        for i in range(0, len(todo), 50):
            r = api.videos().list(part="snippet", id=",".join(todo[i:i+50]),
                                  maxResults=50).execute()
            for it in r.get("items", []):
                sn = it["snippet"]
                f.write(json.dumps({"id": it["id"], "description": sn.get("description", ""),
                                    "tags": sn.get("tags", [])}) + "\n")
            f.flush()

desc = pd.DataFrame([json.loads(l) for l in DESC_CACHE.open() if l.strip()]).drop_duplicates("id")
print(f"{len(desc):,} descriptions on file")

### What the descriptions actually contain

Worth checking before treating them as a content signal.

In [ ]:
PROMO = re.compile(r"(?i)(sponsor|thanks to|check out|buy |discount|promo|use code|affiliate|"
                   r"amazon|newegg|merch|lttstore|floatplane|subscribe|follow us|instagram|"
                   r"twitter|tiktok|facebook|discord|forum|https?://|►|▬|-{5,})")

def promo_share(t):
    lines = [l for l in str(t).split("\n") if len(l.strip()) > 10]
    return np.nan if not lines else sum(bool(PROMO.search(l)) for l in lines)/len(lines)

d = desc.assign(promo=desc.description.map(promo_share),
                chars=desc.description.str.len())
print(f"Median length {d.chars.median():,.0f} characters")
print(f"Share of lines matching promotional patterns: median {d.promo.median():.0%}")
print(f"  above 80%: {(d.promo > .8).mean():.0%} of videos")
print(f"  below 50%: {(d.promo < .5).mean():.0%} of videos")
print("\nA typical description is a sponsor link, one or two sentences about the video,")
print("a forum link and a block of social handles. The substantive part is short.")
print("Descriptions are primarily a monetisation surface, not a content summary —")
print("which is itself an observation about how the business uses them.")

## 3. Transcripts

Captions are the only representation of what a video actually says. They are public but not
straightforwardly collectable, and the route matters.

**What did not work, and why it is worth recording:**

- The official `captions` endpoint needs OAuth *and channel ownership*. Not available to anyone
  outside the company.
- `youtube-transcript-api` reads the public timedtext endpoint directly. It succeeded on a probe
  of twenty and was then refused for the next 2,745 requests. Note the shape of that failure: the
  143 that did land were all from 2021, because the queue ran in date order. Coverage after the
  decline was zero — a corpus that cannot answer the question it was gathered for.
- `--cookies-from-browser` hangs on macOS until the Keychain prompt is answered, and sends a
  logged-in session with every request. Not used.

**What worked:** `yt-dlp` requests the same captions through the player response and can present
a browser TLS fingerprint via `curl-cffi`. But the decisive factor was neither library nor flag —
**it was changing network**. The address was the constraint the whole time.

### Sampling

Rather than attempt everything, take 75 videos from each channel × window cell, and **shuffle the
fetch order across cells**. If the run stops early, what remains is a smaller balanced sample
instead of one complete period and one empty one. That is the specific failure of the first
attempt, fixed by ordering rather than by volume.

In [ ]:
SAMPLE_CSV = DATA/"transcript_sample.csv"
PER_CELL = 75

if not SAMPLE_CSV.exists():
    s = pd.concat([g.sample(min(PER_CELL, len(g)), random_state=42)
                   for _, g in target.groupby(["channel", "window"])])
    s = s.sample(frac=1, random_state=7).reset_index(drop=True)   # shuffle across cells
    s[["id", "channel", "window", "date", "views", "rbi"]].to_csv(SAMPLE_CSV, index=False)

sample = pd.read_csv(SAMPLE_CSV)
print(sample.pivot_table(index="channel", columns="window", values="id",
                         aggfunc="count", fill_value=0).to_string())
print(f"\n{len(sample)} videos sampled, fetch order shuffled")
print(f"Sample vs population — median rbi {sample.rbi.median():.2f} "
      f"vs {target.rbi.median():.2f}")

In [ ]:
# Set OFFLINE = True to re-run this notebook without touching the network at all.
# The corpus is already on disk; nothing downstream needs a fetch. Leaving this on
# by default means a stray Run All can never re-scrape or burn the address.
OFFLINE     = True

MAX_MINUTES = 90
DELAY       = (2.0, 4.0)
IMPERSONATE = True          # needs curl-cffi

assert shutil.which("yt-dlp"), "yt-dlp not on PATH:  %pip install -q yt-dlp curl-cffi"
scratch = pathlib.Path(tempfile.mkdtemp())

def json3_to_text(p):
    d = json.loads(pathlib.Path(p).read_text())
    return " ".join(s.get("utf8", "") for e in d.get("events", []) if "segs" in e
                    for s in e["segs"]).replace("\n", " ").strip()

def fetch_one(vid, timeout=90):
    """Returns (text, status). Refusal and absence are different outcomes and
    are kept separate — only one of them is information about the video."""
    cmd = ["yt-dlp", "--skip-download", "--write-auto-sub", "--write-sub",
           "--sub-lang", "en.*", "--sub-format", "json3", "--no-warnings",
           "-o", str(scratch/"%(id)s"), f"https://youtu.be/{vid}"]
    if IMPERSONATE:
        cmd[1:1] = ["--impersonate", "chrome"]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, "timeout"
    files = sorted(scratch.glob(f"{vid}*.json3"))
    if files:
        text = json3_to_text(files[0])
        for f in files:
            f.unlink()
        return (text, "ok") if len(text) > 200 else (text, "too_short")
    err = (r.stderr or "") + (r.stdout or "")
    if "429" in err or "Too Many Requests" in err:
        return None, "blocked"
    if "no subtitles" in err.lower():
        return None, "no_captions"
    return None, "error"

# Refusals are never cached — those IDs stay eligible for a later attempt.
have = cached_ids(TR_CACHE, statuses={"ok", "no_captions", "too_short"})
todo = [i for i in sample.id if i not in have]
print(f"{len(have)} resolved · {len(todo)} remaining\n")

if OFFLINE:
    print("OFFLINE = True — no requests will be made.")
    print(f"{len(have)} of {len(sample)} sampled videos already resolved on disk.")
    todo = []

probe = [fetch_one(v)[1] for v in todo[:3]] if todo else []
if todo:
    print(f"Probe: {probe}")
    todo = todo[3:]          # the probe already resolved these; don't fetch them twice

if probe.count("blocked") >= 2:
    print("\nRefused on this connection. Switch networks and re-run this cell.")
elif todo:
    t0, done, ok_n, streak, stats = time.time(), 0, 0, 0, {}
    with TR_CACHE.open("a") as f:
        for vid in todo:
            if (time.time()-t0)/60 >= MAX_MINUTES:
                print(f"\nReached the {MAX_MINUTES}-minute budget."); break
            time.sleep(random.uniform(*DELAY))
            text, status = fetch_one(vid)
            stats[status] = stats.get(status, 0) + 1
            if status in ("blocked", "timeout", "error"):
                streak += 1
                if streak >= 5:
                    print(f"\nFive consecutive failures — stopping to preserve the address.")
                    break
                continue
            streak = 0
            f.write(json.dumps({"id": vid, "transcript": text, "status": status}) + "\n")
            f.flush(); done += 1; ok_n += (status == "ok")
            if done % 25 == 0:
                mins = (time.time()-t0)/60
                print(f"  {done} resolved · {ok_n} with text · {mins:.0f}m · {stats}")
    print(f"\nStopped after {(time.time()-t0)/60:.0f} minutes · {stats}")

## 4. Coverage

Balance across cells decides whether the corpus can support a comparison. A cell at zero cannot be
compensated for by volume elsewhere.

In [ ]:
if TR_CACHE.exists():
    # keep="last" matters: a video refused on an early attempt and retrieved on a later
    # one appears twice. Keeping the first record would report the refusal and understate
    # coverage — it reported 27% instead of the actual 99.7%.
    tr = (pd.DataFrame([json.loads(l) for l in TR_CACHE.open() if l.strip()])
            .drop_duplicates("id", keep="last"))
    tr["chars"] = tr.transcript.fillna("").str.len()
    got = set(tr[tr.status == "ok"].id)
    sample["have"] = sample.id.isin(got)

    print(f"{sample.have.sum()} of {len(sample)} sampled videos have a transcript "
          f"({sample.have.mean():.0%})\n")
    print(sample.pivot_table(index="channel", columns="window", values="have",
                             aggfunc=["sum", "mean"]).round(2).to_string())

    if sample.have.sum() >= 40:
        s_ok = sample[sample.have]
        print(f"\nRetrieved subset vs full sample — checking retrieval is not itself selective:")
        print(f"  median rbi    {s_ok.rbi.median():.2f}  vs  {sample.rbi.median():.2f}")
        print(f"  median views  {s_ok.views.median():,.0f}  vs  {sample.views.median():,.0f}")
        t_ok = tr[tr.status == "ok"]
        print(f"\nCorpus: {len(t_ok)} transcripts, median {t_ok.chars.median():,.0f} characters")
        print(f"Against a fifty-character title, roughly {t_ok.chars.median()/50:.0f}x more text.")
else:
    print("No transcripts yet.")

In [ ]:
out = target[["id", "channel", "window", "title", "date", "views", "likes",
              "comments", "duration_min", "rbi"]].copy()
out = out.merge(desc[["id", "description"]], on="id", how="left")
if TR_CACHE.exists():
    out = out.merge(tr[["id", "transcript", "status"]], on="id", how="left")
out.to_csv(DATA/"content_corpus.csv", index=False)
print(f"Wrote content_corpus.csv — {len(out):,} rows")
print(out.notna().mean().round(3).to_string())

---

## What this buys, and what it does not

**Buys.** A representation of what a video is about rather than how it was labelled. Transcripts
run to roughly nine thousand characters against a title's fifty.

**Does not buy.** The separation of content from distribution. That needs click-through rate,
average view duration and retention — the variables sitting between a video being made and a video
being pushed — and all three are visible only to the channel owner. Better content features
describe what was made; they do not identify what the algorithm did with it.

**Also does not buy.** Production quality, host performance, editing, or whether the subject was
inherently interesting. A transcript records what was said, not how well.

**A note on the collection itself.** Three routes to the same public data behaved completely
differently: an official endpoint that is closed to non-owners, a direct client that was refused
after 160 requests, and a tool that worked once the network changed. "Publicly available" and
"collectable at scale" are not the same property, and which one applies is worth establishing
before building an analysis that depends on it.